## 01 — Purpose and reproducibility contract


**Paper-01 Phase-1:** build one immutable, event-level DAS benchmark from boolean annotation masks.  
The notebook extracts 8-connected `(time-window × fiber-location)` components, creates leakage-resistant B1/B2/B3 benchmark tracks, runs integrity tests, verifies deterministic rebuilds, and freezes the benchmark with SHA-256.

**Contract:** downstream Phase-2+ notebooks read the frozen CSV files; they never regenerate event definitions or splits. Raw signal payload is intentionally not required in Phase-1.

In [ ]:
NOTEBOOK_VERSION = "Paper01_PHASE1_MASTER_v0.1.4"
BENCHMARK_FOLDER = "Phase1_Benchmark_v0.1"
AUTO_DOWNLOAD_MASKS = True
TRANSPORT_REPO_ID = "salah48484848/DAS-dataset"
N_LOCATION_BINS = 20
MASK_FILES = [
    "car/auto2_2023-04-17T124510+0100.npy",
    "car/auto_2023-04-17T124152+0100.npy",
    "car/auto4_2023-06-06T144655+0100.npy",
    "car/auto4_2023-06-06T145055+0100.npy",
    "construction/reconstruction1_2023-06-06T150714+0100.npy",
    "fence/fence_2023-04-17T123117+0100.npy",
    "fence/kopani_plot_2023-11-07T150207Z.npy",
    "fence/plot_2023-11-07T150119Z.npy",
    "longboard/longboard_2023-04-17T123543+0100.npy",
    "longboard/longboardroad_2023-04-17T123638+0100.npy",
    "manipulation/manipulation_2023-04-17T122847+0100.npy",
    "openclose/openingclosing_2023-04-17T122658+0100.npy",
    "regular/regularshit_2023-04-17T130103+0100.npy",
    "running/running_2023-04-17T122413+0100.npy",
    "walk/walking2_2023-04-17T122153+0100.npy",
    "walk/walking_2023-04-17T121933+0100.npy",
]


## 02 — Global configuration

All benchmark constants are frozen in `BenchmarkConfig`; changing any value changes the final benchmark fingerprint.

In [ ]:
EXPECTED = {
    "benchmark_version": "Phase1_v0.1",
    "connectedness": 8,
    "window_size": 8192,
    "window_shift": 2048,
    "temporal_guard_windows": 5,
    "reference_split": (0.80, 0.10, 0.10),
    "n_spatial_folds": 4,
    "min_transfer_events_per_class": 3,
    "min_transfer_classes": 2,
    "min_transfer_source_groups_per_class": 2,
    "random_seed": 42,
    "code_version": "Paper01_PHASE1_MASTER_v0.1.4",
}
print(EXPECTED)


## 03 — Environment and dependencies

Installs only Phase-1 dependencies plus the lightweight transport client for the annotation masks.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "pandas>=2.1", "scipy>=1.11", "matplotlib>=3.8",
    "pytest>=8.0", "nbformat>=5.9", "huggingface_hub>=0.24"
])
print("Environment ready")

## 04 — Google Drive mount and project root

Mount Drive in Colab, locate the bundled `phase1_benchmark` package, create the benchmark scaffold, and import the reusable implementation.

In [ ]:
from pathlib import Path
import os, sys, base64, io, zipfile

# Self-contained fallback: exact phase1_benchmark package bundled with this notebook.
EMBEDDED_PACKAGE_B64 = '...'

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    # Avoid a redundant mount call when Drive is already available.
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_PARENT = Path("/content/drive/MyDrive/Project_Paper01_SCAL_DAS")
else:
    PROJECT_PARENT = Path.cwd()
PROJECT_PARENT.mkdir(parents=True, exist_ok=True)
BENCH_ROOT = Path(os.environ.get("PHASE1_ROOT", PROJECT_PARENT / BENCHMARK_FOLDER))

EXPECTED_CODE_VERSION = EXPECTED["code_version"]

def _find_package_root(candidates):
    for p in candidates:
        p = Path(p)
        cfg_path = p / "phase1_benchmark" / "config.py"
        if cfg_path.is_file():
            try:
                if EXPECTED_CODE_VERSION in cfg_path.read_text(encoding="utf-8"):
                    return p
            except Exception:
                pass
    return None

# 1) Reuse an already extracted package if present.
package_candidates = [Path.cwd(), Path("/content/phase1_package"), PROJECT_PARENT]
if Path("/content").exists():
    package_candidates.extend(p.parent for p in Path("/content").glob("*/phase1_benchmark"))
package_root = _find_package_root(package_candidates)
bootstrap_source = "existing-package" if package_root else None

# 2) If the full Phase-1 ZIP was uploaded to Colab/Drive, extract it automatically.
if package_root is None:
    zip_candidates = [
        Path.cwd() / "Paper01_PHASE1_v0.1.4_PACKAGE.zip",
        Path("/content/Paper01_PHASE1_v0.1.4_PACKAGE.zip"),
        PROJECT_PARENT / "Paper01_PHASE1_v0.1.4_PACKAGE.zip",
    ]
    archive = next((p for p in zip_candidates if p.is_file()), None)
    if archive is not None:
        extract_root = Path("/content/phase1_package") if IN_COLAB else Path.cwd() / ".phase1_package"
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(extract_root)
        extracted_candidates = [extract_root]
        extracted_candidates.extend(p.parent for p in extract_root.glob("*/phase1_benchmark"))
        package_root = _find_package_root(extracted_candidates)
        bootstrap_source = f"archive:{archive}"

# 3) Final fallback: materialize the exact bundled package from this notebook.
if package_root is None:
    extract_root = Path("/content/phase1_embedded") if IN_COLAB else Path.cwd() / ".phase1_embedded"
    extract_root.mkdir(parents=True, exist_ok=True)
    raw = base64.b64decode(EMBEDDED_PACKAGE_B64)
    with zipfile.ZipFile(io.BytesIO(raw), "r") as zf:
        zf.extractall(extract_root)
    package_root = _find_package_root([extract_root])
    bootstrap_source = "embedded-notebook-package"

if package_root is None:
    raise RuntimeError("Phase-1 package bootstrap failed unexpectedly.")

sys.path.insert(0, str(package_root))

from phase1_benchmark.config import BenchmarkConfig
from phase1_benchmark.io_utils import create_scaffold, write_event_registry, write_split_artifacts, write_audit_artifacts, write_manifest_and_fingerprint
from phase1_benchmark.components import extract_components, sha256_file
from phase1_benchmark.splits import build_reference_split, build_spatiotemporal_leakage_groups, assign_spatial_bins, build_spatial_transfer_splits
from phase1_benchmark.audit import audit_event_registry, audit_reference_split, audit_spatial_transfer
from phase1_benchmark.freeze import (
    build_manifest, benchmark_fingerprint, verify_frozen_benchmark,
    build_persisted_file_hashes, verify_persisted_file_hashes, persisted_benchmark_fingerprint,
)
from phase1_benchmark.figures import plot_event_component_examples, plot_class_split_distribution, plot_spatial_transfer_map

cfg = BenchmarkConfig()
for key, value in EXPECTED.items():
    assert getattr(cfg, key) == value, (key, getattr(cfg, key), value)
paths = create_scaffold(BENCH_ROOT, cfg)
print("Benchmark root:", BENCH_ROOT)
print("Package root:", package_root)
print("Package bootstrap:", bootstrap_source)


## 05 — Input discovery and Phase-0B artifacts

If no local masks are present, download only the small boolean annotation masks from the transport mirror. The canonical dataset citation remains the Scientific Data/Figshare release.

In [ ]:
from huggingface_hub import hf_hub_download

missing_mask_files = [fn for fn in MASK_FILES if not (paths["inputs_masks"] / fn).exists()]
if AUTO_DOWNLOAD_MASKS:
    for fn in missing_mask_files:
        hf_hub_download(repo_id=TRANSPORT_REPO_ID, filename=fn, repo_type="dataset", local_dir=str(paths["inputs_masks"]))
existing_masks = sorted(paths["inputs_masks"].rglob("*.npy"))
remaining_missing = [fn for fn in MASK_FILES if not (paths["inputs_masks"] / fn).exists()]
if remaining_missing:
    raise FileNotFoundError(f"Required annotation masks are still missing: {remaining_missing}")
print("Annotation masks:", len(existing_masks))

phase0b_files = sorted(p for p in paths["inputs_phase0b"].iterdir() if p.is_file())
print("Phase-0B artifacts discovered:", [p.name for p in phase0b_files])

## 06 — Source-mask checksum inventory

Create a stable inventory keyed by logical class/file path; filesystem mount paths are not benchmark identities.

In [ ]:
import re
import pandas as pd

source_rows = []
for p in existing_masks:
    rel = p.relative_to(paths["inputs_masks"]).as_posix()
    class_name = p.parent.name
    recording_id = p.stem
    m = re.search(r"(20\d{2}-\d{2}-\d{2})", recording_id)
    source_rows.append({
        "class_name": class_name,
        "recording_id": recording_id,
        "recording_date": m.group(1) if m else None,
        "source_mask_file": rel,
        "source_mask_sha256": sha256_file(p),
        "mask_path": str(p),
    })
source_inventory = pd.DataFrame(source_rows).sort_values(["class_name", "recording_id"], kind="stable").reset_index(drop=True)
assert source_inventory["recording_id"].is_unique
source_inventory.head()

## 07 — Structural validation

Validate every annotation array before connected-component extraction; record the exact dimensions and dtype.

In [ ]:
import numpy as np

structure_rows = []
for row in source_inventory.itertuples(index=False):
    arr = np.load(row.mask_path, mmap_mode="r")
    if arr.ndim != 2:
        raise ValueError(f"{row.source_mask_file}: expected 2-D mask, got {arr.shape}")
    if arr.shape[1] < 2:
        raise ValueError(f"{row.source_mask_file}: invalid spatial dimension {arr.shape[1]}")
    if arr.dtype != np.bool_:
        raise ValueError(f"{row.source_mask_file}: expected boolean mask, got {arr.dtype}")
    structure_rows.append({
        "class_name": row.class_name,
        "recording_id": row.recording_id,
        "n_windows": int(arr.shape[0]),
        "n_spatial": int(arr.shape[1]),
        "dtype": str(arr.dtype),
        "positive_cells": int(np.asarray(arr).sum()),
    })
structure_df = pd.DataFrame(structure_rows)
print(structure_df.groupby("n_spatial").size().rename("recordings"))
structure_df

## 08 — 8-connected component extraction

Extract indivisible event candidates from every `(window × spatial-channel)` mask.

In [ ]:
class_names = sorted(source_inventory["class_name"].unique().tolist())
class_to_id = {name: i for i, name in enumerate(class_names)}
component_frames = []
for row in source_inventory.itertuples(index=False):
    mask = np.load(row.mask_path, mmap_mode="r")
    frame = extract_components(
        mask=np.asarray(mask),
        class_name=row.class_name,
        class_id=class_to_id[row.class_name],
        recording_id=row.recording_id,
        recording_date=row.recording_date,
        source_mask_file=row.source_mask_file,
        source_mask_sha256=row.source_mask_sha256,
        config=cfg,
    )
    component_frames.append(frame)
print("Recordings processed:", len(component_frames))

## 09 — Event registry generation

Concatenate and deterministically order all physical-event components; `EVENT_REGISTRY.csv` is the source of truth.

In [ ]:
event_registry = pd.concat(component_frames, ignore_index=True) if component_frames else pd.DataFrame()
event_registry = event_registry.sort_values(
    ["class_name", "recording_id", "window_start", "channel_start", "component_index", "event_id"],
    kind="stable",
).reset_index(drop=True)
if event_registry["event_id"].duplicated().any():
    raise RuntimeError("Deterministic event_id collision detected")
print("Event components:", len(event_registry))
event_registry.head()

## 10 — Component statistics and exclusions

No arbitrary area threshold is applied. Components are retained unless an explicit integrity rule excludes them.

In [ ]:
component_stats = (
    event_registry.groupby(["class_name", "recording_id"], dropna=False)
    .agg(n_components=("event_id", "nunique"), total_area_cells=("component_area_cells", "sum"), median_area_cells=("component_area_cells", "median"))
    .reset_index()
)
excluded_components = event_registry[event_registry["candidate_status"] != "included"].copy()
print("Excluded components:", len(excluded_components))
component_stats

## 11 — B1 spatio-temporal leakage-group split

**v0.1.3 rule:** a temporal overlap alone is not treated as leakage in a distributed fiber sensor. Two connected events are placed in the same atomic leakage group only when they come from the same recording, have fewer than 5 blank windows between their temporal intervals, **and** their raw fiber-channel intervals overlap. Connected conflict groups are then split deterministically within each class as indivisible units, targeting 80/10/10 by event count while requiring train/validation/test coverage.

This rule is data-driven and was validated on the real 137-event Phase-1 registry: all 9 classes have at least 5 independent leakage groups, making full B1 train/val/test class coverage feasible without spatially false temporal exclusions.


In [ ]:
leakage_groups = build_spatiotemporal_leakage_groups(event_registry, cfg)
leakage_group_stats = (
    event_registry[["event_id", "class_name"]]
    .merge(leakage_groups, on="event_id", how="left")
    .groupby("class_name")
    .agg(n_events=("event_id", "nunique"), n_leakage_groups=("leakage_group_id", "nunique"))
)
print("B1 spatio-temporal leakage groups:")
display(leakage_group_stats)

reference_split = build_reference_split(event_registry, cfg)
print(reference_split["split"].value_counts(dropna=False))
b1_table = pd.crosstab(reference_split["class_name"], reference_split["split"])
display(b1_table)
missing_b1 = {
    role: sorted(set(event_registry["class_name"]) - set(reference_split.loc[reference_split["split"] == role, "class_name"]))
    for role in ["train", "val", "test"]
}
print("B1 missing-class diagnostic:", missing_b1)


## 12 — B2 normalized spatial-bin labels

Create 20 normalized-location labels for later shortcut probes without altering B1 partitions.

In [ ]:
location_labels = assign_spatial_bins(event_registry[["event_id", "class_name", "u_centroid"]].copy(), N_LOCATION_BINS)
location_labels = location_labels.rename(columns={"spatial_fold": "location_bin_20"})
print(location_labels["location_bin_20"].value_counts().sort_index())

## 13 — B3 far-region spatial transfer split

Phase-0B cell counts overstated independent support because many positive cells belong to the same connected physical event. The real event-level audit gives Fold-1 ↔ Fold-4 common support for **Car** and **Construction** at a minimum of 3 events per class per side. Therefore v0.1.2 freezes `MIN_TRANSFER_EVENTS_PER_CLASS=3` and requires at least **2 shared classes**. B3 is an exploratory far-region transfer benchmark and must be reported with event-level uncertainty/CI later; it is not a 9-class transfer claim.


In [ ]:
spatial_transfer = build_spatial_transfer_splits(event_registry, cfg)
print("Transfer rows:", len(spatial_transfer))
print(
    "B3 support rule:", cfg.min_transfer_events_per_class, "events/class/side;",
    cfg.min_transfer_source_groups_per_class, "independent source leakage groups/class;",
    cfg.min_transfer_classes, "shared classes minimum"
)
if not spatial_transfer.empty:
    display(pd.crosstab(spatial_transfer["transfer_pair_id"], spatial_transfer["transfer_role"]))
    eligible_summary = spatial_transfer[spatial_transfer["transfer_role"].isin(["source_train", "source_val", "target_test"])].groupby(["transfer_pair_id", "class_name", "transfer_role"])["event_id"].nunique().unstack(fill_value=0)
    display(eligible_summary)
else:
    print("No valid B3 pair under the frozen event-level support rule. Integrity audit will stop the freeze.")


## 14 — Integrity and leakage audit

Every mandatory gate must PASS before any final benchmark freeze file is written. In v0.1.3 this includes an explicit **spatio-temporal leakage-group atomicity gate** in addition to event-ID disjointness, 9-class train/validation/test coverage, deterministic rebuild, and B3 transfer checks.


In [ ]:
registry_audit = audit_event_registry(event_registry, source_inventory[["source_mask_file", "source_mask_sha256"]])
reference_audit = audit_reference_split(reference_split, event_registry, cfg)
transfer_audit = audit_spatial_transfer(spatial_transfer, event_registry, cfg)
registry_audit.insert(0, "track", "registry")
reference_audit.insert(0, "track", "B1_reference")
transfer_audit.insert(0, "track", "B3_transfer")
audit_df = pd.concat([registry_audit, reference_audit, transfer_audit], ignore_index=True)
display(audit_df)
if (audit_df["status"] != "PASS").any():
    raise RuntimeError("Mandatory benchmark integrity gate failed; inspect audit_df before proceeding")

## 15 — Synthetic unit tests

Run the standalone regression suite bundled with this Phase-1 package.

In [ ]:
import os, subprocess, sys

test_dir = package_root / "tests"
if not test_dir.is_dir():
    raise FileNotFoundError(f"Standalone tests not found at {test_dir}")

# IMPORTANT:
# sys.path changes in the notebook kernel are not automatically inherited
# by a new Python subprocess. Make the package importable explicitly.
env = os.environ.copy()
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = (
    str(package_root)
    if not existing_pythonpath
    else str(package_root) + os.pathsep + existing_pythonpath
)

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=str(package_root),
    env=env,
    text=True,
    capture_output=True,
)

print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Standalone Phase-1 unit tests failed")

UNIT_TESTS_PASS = True
print("UNIT_TESTS_PASS =", UNIT_TESTS_PASS)


## 16 — Dataset-derived consistency tests

Compare the current masks with Phase-0B exact-structure output when that artifact is available; otherwise retain a documented SKIP rather than inventing a result.

In [ ]:
consistency_rows = []
exact_path = paths["inputs_phase0b"] / "NPY_EXACT_STRUCTURE.csv"
if exact_path.exists():
    exact = pd.read_csv(exact_path, encoding="utf-8-sig")
    merged = structure_df.merge(exact[["recording_id", "n_windows", "n_spatial"]], on="recording_id", how="left", suffixes=("_now", "_phase0b"))
    merged["shape_match"] = (merged["n_windows_now"] == merged["n_windows_phase0b"]) & (merged["n_spatial_now"] == merged["n_spatial_phase0b"])
    if not merged["shape_match"].all():
        raise RuntimeError("Current annotation-mask structure differs from Phase-0B audit")
    consistency_rows.append({"check": "phase0b_exact_structure", "status": "PASS", "detail": f"{len(merged)} recordings matched"})
else:
    consistency_rows.append({"check": "phase0b_exact_structure", "status": "SKIP", "detail": "NPY_EXACT_STRUCTURE.csv not supplied"})

consistency_rows.append({"check": "event_id_unique", "status": "PASS" if event_registry["event_id"].is_unique else "FAIL", "detail": f"events={len(event_registry)}"})
consistency_df = pd.DataFrame(consistency_rows)
display(consistency_df)
if (consistency_df["status"] == "FAIL").any():
    raise RuntimeError("Dataset-derived consistency test failed")

## 17 — Figures and summary tables

Write diagnostic artifacts only after the mandatory integrity and unit-test gates have passed.

In [ ]:
assert UNIT_TESTS_PASS
write_event_registry(event_registry, paths)
write_split_artifacts(reference_split, spatial_transfer, paths, events=event_registry)
write_audit_artifacts(audit_df, paths)
location_labels.to_csv(paths["registry"] / "LOCATION_LABELS_20.csv", index=False)
structure_df.to_csv(paths["registry"] / "SOURCE_STRUCTURE.csv", index=False)
consistency_df.to_csv(paths["audits"] / "DATASET_CONSISTENCY.csv", index=False)

plot_event_component_examples(event_registry, paths["figures"] / "event_components_examples.png")
plot_class_split_distribution(reference_split, paths["figures"] / "class_split_distribution.png")
plot_spatial_transfer_map(spatial_transfer, paths["figures"] / "spatial_transfer_map.png")
print("Registry/split/audit tables and figures written")

## 18 — Deterministic in-memory rebuild fingerprint

Build an in-memory reproducibility fingerprint from the event registry and split tables. This fingerprint is used only to prove deterministic rebuild behavior; it is **not** the portable final benchmark SHA.


In [ ]:
freeze_inventory = source_inventory[["source_mask_file", "source_mask_sha256"]].copy()
in_memory_candidate_sha = benchmark_fingerprint(
    cfg, freeze_inventory, event_registry, reference_split, spatial_transfer
)
if len(in_memory_candidate_sha) != 64:
    raise RuntimeError("Invalid in-memory SHA-256 length")
print("In-memory deterministic fingerprint:", in_memory_candidate_sha)
print("Counts:", {
    "events": int(len(event_registry)),
    "reference_rows": int(len(reference_split)),
    "transfer_rows": int(len(spatial_transfer)),
})


## 19 — Portable SHA-256 freeze from persisted artifact bytes

Hash the exact CSV/NPY artifacts already written to disk, create `BENCHMARK_FILE_HASHES.json`, and derive the final benchmark SHA from the canonical ordered file-hash map plus frozen configuration.


In [ ]:
frozen_relative_paths = []
for p in sorted(paths["inputs_masks"].rglob("*.npy")):
    frozen_relative_paths.append(p.relative_to(paths["root"]).as_posix())

for rel in [
    "registry/EVENT_REGISTRY.csv",
    "registry/COMPONENT_STATS.csv",
    "registry/EXCLUDED_COMPONENTS.csv",
    "registry/LOCATION_LABELS_20.csv",
    "registry/SOURCE_STRUCTURE.csv",
    "splits/TRAIN_VAL_TEST.csv",
    "splits/SPATIAL_TRANSFER_SPLITS.csv",
    "splits/CLASS_SPLIT_STATS.csv",
    "audits/LEAKAGE_AUDIT.csv",
    "audits/DATASET_CONSISTENCY.csv",
]:
    if (paths["root"] / rel).is_file():
        frozen_relative_paths.append(rel)

file_hashes = build_persisted_file_hashes(paths["root"], frozen_relative_paths)
manifest = build_manifest(
    cfg, freeze_inventory, event_registry, reference_split, spatial_transfer,
    file_hashes=file_hashes,
)
write_manifest_and_fingerprint(manifest, paths)
candidate_sha = manifest["benchmark_sha256"]
verify_persisted_file_hashes(paths["root"], file_hashes)
assert persisted_benchmark_fingerprint(cfg, file_hashes) == candidate_sha
print("Persisted frozen artifacts:", len(file_hashes))
print("Portable benchmark SHA-256:", candidate_sha)


## 20 — Deterministic rebuild + persisted freeze verification

Re-extract connected components from the source masks, rebuild B1/B3, verify the in-memory deterministic fingerprint, then independently verify every persisted artifact hash and the final portable benchmark SHA.


In [ ]:
rebuild_frames = []
for row in source_inventory.itertuples(index=False):
    mask = np.load(row.mask_path, mmap_mode="r")
    rebuild_frames.append(extract_components(
        mask=np.asarray(mask), class_name=row.class_name, class_id=class_to_id[row.class_name],
        recording_id=row.recording_id, recording_date=row.recording_date,
        source_mask_file=row.source_mask_file, source_mask_sha256=row.source_mask_sha256, config=cfg,
    ))
rebuilt_events = pd.concat(rebuild_frames, ignore_index=True).sort_values(
    ["class_name", "recording_id", "window_start", "channel_start", "component_index", "event_id"], kind="stable"
).reset_index(drop=True)
rebuilt_reference = build_reference_split(rebuilt_events, cfg)
rebuilt_transfer = build_spatial_transfer_splits(rebuilt_events, cfg)
rebuilt_sha = benchmark_fingerprint(cfg, freeze_inventory, rebuilt_events, rebuilt_reference, rebuilt_transfer)
if rebuilt_sha != in_memory_candidate_sha:
    raise RuntimeError(f"Deterministic rebuild mismatch: {in_memory_candidate_sha} != {rebuilt_sha}")

# Portable verification reads exact persisted bytes and source masks.
verify_frozen_benchmark(
    manifest,
    source_root=paths["inputs_masks"],
    benchmark_root=paths["root"],
    config=cfg, source_inventory=freeze_inventory,
    events=event_registry, reference_split=reference_split, transfer_split=spatial_transfer,
)

# Re-read the persisted hash map to prove portability independent of DataFrame float round-trips.
manifest_on_disk = json.loads((paths["root"] / "BENCHMARK_MANIFEST.json").read_text(encoding="utf-8"))
verify_persisted_file_hashes(paths["root"], manifest_on_disk["file_hashes"])
portable_recomputed = persisted_benchmark_fingerprint(cfg, manifest_on_disk["file_hashes"])
if portable_recomputed != candidate_sha:
    raise RuntimeError(f"Portable frozen SHA mismatch: {candidate_sha} != {portable_recomputed}")

DETERMINISTIC_REBUILD_PASS = True
print("Deterministic rebuild PASS:", rebuilt_sha)
print("Persisted-byte freeze verification PASS:", portable_recomputed)


## 21 — Export frozen benchmark

Create a portable ZIP of the frozen benchmark directory after all mandatory gates have passed.

In [ ]:
import shutil
assert DETERMINISTIC_REBUILD_PASS
zip_base = BENCH_ROOT.parent / f"{BENCHMARK_FOLDER}_FROZEN"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=BENCH_ROOT))
print("Frozen benchmark ZIP:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

## 22 — Final PASS/FAIL report

A PASS is emitted only when unit tests, integrity gates, source verification, and deterministic rebuild all succeeded.

In [ ]:
FINAL_PASS = bool(
    UNIT_TESTS_PASS
    and DETERMINISTIC_REBUILD_PASS
    and (audit_df["status"] == "PASS").all()
    and not (consistency_df["status"] == "FAIL").any()
)
summary = {
    "benchmark_version": cfg.benchmark_version,
    "code_version": cfg.code_version,
    "status": "PASS" if FINAL_PASS else "FAIL",
    "event_components": int(len(event_registry)),
    "reference_split_rows": int(len(reference_split)),
    "transfer_rows": int(len(spatial_transfer)),
    "benchmark_sha256": candidate_sha,
    "frozen_zip": str(zip_path),
}
print(summary)
if not FINAL_PASS:
    raise RuntimeError("Phase-1 benchmark did not satisfy all mandatory gates")